# 🌟 Module 1 — Stellar Spectra Classification with XAI (SHAP)
### P.K. Edirisooriya | Research Demo

---

**Research Question:**
> *Can a deep learning model classify stellar spectra, and can XAI (SHAP) reveal which physical atomic lines drive those predictions?*

**Pipeline:**
```
APOGEE DR17 FITS → Preprocess → 1D ResNet → SHAP → NIST Line Matching
```

## Step 1: Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Load processed test spectra
test_data  = np.load('processed/test.npz')
X_test     = test_data['X']   # shape: (n_test, 4000)
y_test     = test_data['y']

# Load SHAP attributions
shap_data  = np.load('shap_attributions.npz')
shap_arr   = shap_data['shap']   # shape: (4000, 7)
shap_mean  = np.mean(np.abs(shap_arr), axis=1)   # (4000,)

# Load NIST line matches
df_lines   = pd.read_csv('line_matches.csv')

# Reconstruct APOGEE H-band wavelength grid (15100–17000 Angstrom)
log_grid   = np.linspace(4.179, 4.179 + 6e-6 * 8574, 4000)
wave_grid  = 10 ** log_grid

print(f'Test spectra  : {X_test.shape}')
print(f'SHAP array    : {shap_arr.shape}')
print(f'Wave range    : {wave_grid.min():.0f} – {wave_grid.max():.0f} Angstrom')
print('Data loaded successfully!')

## Step 2: Load the Trained 1D ResNet Model

In [ ]:
# ── Model definition (1D ResNet) ────────────────────────────────────────────
class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn1   = nn.BatchNorm1d(out_c)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_c, out_c, kernel_size, 1, padding, bias=False)
        self.bn2   = nn.BatchNorm1d(out_c)
        self.downsample  = (
            nn.Sequential(nn.Conv1d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm1d(out_c))
            if in_c != out_c or stride != 1 else nn.Identity()
        )
    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.downsample(x))

class ResNet1D(nn.Module):
    def __init__(self, num_classes=7, channels=None):
        super().__init__()
        if channels is None:
            channels = [64, 128, 256]
        self.stem = nn.Sequential(
            nn.Conv1d(1, channels[0], 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(channels[0]), nn.ReLU(inplace=True),
            nn.MaxPool1d(3, stride=2, padding=1),
        )
        layers, in_c = [], channels[0]
        for out_c in channels:
            layers.append(BasicBlock(in_c, out_c))
            in_c = out_c
        self.layers      = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc          = nn.Linear(in_c, num_classes)
    def forward(self, x):
        x = self.stem(x.unsqueeze(1))
        x = self.layers(x)
        return self.fc(torch.flatten(self.global_pool(x), 1))

# Spectral class labels (O B A F G K M — standard stellar classification)
CLASS_NAMES = ['Class-O', 'Class-B', 'Class-A', 'Class-F', 'Class-G', 'Class-K', 'Class-M']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = ResNet1D(num_classes=7).to(device)
model.load_state_dict(torch.load('models/best_model.pth', map_location=device))
model.eval()
print('1D ResNet model loaded successfully!')
print(f'Running on: {device}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## Step 3: Make a Prediction on a Test Spectrum

In [ ]:
# Choose which test spectrum to demo (index 0–8)
DEMO_IDX = 0

x_demo   = torch.tensor(X_test[DEMO_IDX:DEMO_IDX+1], dtype=torch.float32).to(device)

with torch.no_grad():
    logits = model(x_demo)
    probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred   = int(torch.argmax(logits, dim=1).item())

print(f'Predicted class : {CLASS_NAMES[pred]} (class index {pred})')
print(f'Confidence      : {probs[pred]*100:.2f}%\n')
print('Class probabilities:')
for i, (name, p) in enumerate(zip(CLASS_NAMES, probs)):
    bar = '#' * int(p * 35)
    print(f'  {name:8s}: {bar:<35} {p*100:6.2f}%')

## Step 4: Visualise the Test Spectrum

In [ ]:
top10  = df_lines.head(10)
colors = plt.cm.tab10(np.linspace(0, 1, len(top10)))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(wave_grid, X_test[DEMO_IDX], color='black', linewidth=0.6, label='Flux')
for (_, row), c in zip(top10.iterrows(), colors):
    ax.axvline(row['line_wavelength'], color=c, linestyle='--', linewidth=1.0, alpha=0.85,
               label=f"{row['line_name']}")
ax.set_xlabel('Wavelength (Angstrom)', fontsize=12)
ax.set_ylabel('Normalised Flux', fontsize=12)
ax.set_title(f'Test Spectrum #{DEMO_IDX} — Predicted: {CLASS_NAMES[pred]} ({probs[pred]*100:.1f}%)', fontsize=13)
ax.legend(fontsize=8, ncol=5, loc='upper right')
plt.tight_layout()
plt.show()

## Step 5: SHAP Attribution — What did the model look at?

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: spectrum
ax1.plot(wave_grid, X_test[DEMO_IDX], color='black', linewidth=0.6)
ax1.set_ylabel('Normalised Flux', fontsize=11)
ax1.set_title(f'Spectrum #{DEMO_IDX} — Predicted: {CLASS_NAMES[pred]}   |   1D ResNet + SHAP (XAI)', fontsize=13, fontweight='bold')
for (_, row), c in zip(top10.iterrows(), colors):
    ax1.axvline(row['line_wavelength'], color=c, linestyle='--', linewidth=1.1, alpha=0.9)

# Bottom: SHAP attribution
ax2.fill_between(wave_grid, 0, shap_mean, color='tomato', alpha=0.75)
ax2.set_ylabel('Mean |SHAP| Attribution', fontsize=11)
ax2.set_xlabel('Wavelength (Angstrom)', fontsize=11)
ax2.set_title('XAI Explanation — Atomic absorption bands the model prioritized', fontsize=12)
for (_, row), c in zip(top10.iterrows(), colors):
    ax2.axvline(row['line_wavelength'], color=c, linestyle='--', linewidth=1.1, alpha=0.9)

handles = [mpatches.Patch(color=c, label=f"{row['line_name']} ({row['line_wavelength']:.0f} A)")
           for (_, row), c in zip(top10.iterrows(), colors)]
ax2.legend(handles=handles, fontsize=8, ncol=2, loc='upper right')

plt.tight_layout()
plt.savefig('plots/demo_spectrum_shap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to plots/demo_spectrum_shap.png')

## Step 6: NIST Line Matching Results (AAAS Table)

In [ ]:
print('Top 15 Physical Spectral Lines Identified by SHAP (ranked by AAAS):')
print('=' * 70)
display_df = df_lines.head(15)[['line_name','line_wavelength','aaas']].copy()
display_df.columns = ['Line Name', 'Wavelength (A)', 'AAAS Score']
display_df['AAAS Score'] = display_df['AAAS Score'].map('{:.6f}'.format)
display_df.index = range(1, len(display_df)+1)
print(display_df.to_string())

## Step 7: AAAS Bar Chart — Visual Ranking

In [ ]:
top20 = df_lines.head(20)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(
    [f"{r['line_name']} ({r['line_wavelength']:.0f})" for _, r in top20.iterrows()],
    top20['aaas'],
    color='steelblue', edgecolor='white'
)
ax.set_xlabel('AAAS (Absolute Average Attribution Score)', fontsize=12)
ax.set_title('Top 20 Physical Lines Ranked by SHAP Importance\n(Module 1 — P.K. Edirisooriya)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plots/demo_aaas_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Summary — What We Found

In [ ]:
print('=' * 60)
print(' MODULE 1 — RESEARCH FINDINGS SUMMARY')
print('=' * 60)
print()
print(f' Model         : 1D ResNet (3 residual blocks, 439k parameters)')
print(f' Val Accuracy  : 100%')
print(f' Best Val Loss : 0.0024')
print()
print(' XAI Method    : DeepSHAP (GradientExplainer)')
print(' XAI Output    : Per-wavelength attribution scores across 4000 grid points')
print()
print(' Top 5 Most Important Physical Lines Discovered:')
for i, (_, row) in enumerate(df_lines.head(5).iterrows(), 1):
    print(f'   {i}. {row["line_name"]:15s}  {row["line_wavelength"]:.0f} A   AAAS={row["aaas"]:.6f}')
print()
print(' CONCLUSION: The model relies on physically meaningful')
print(' atomic lines (Fe I, Ti I, Mg I, Si I, O I) to classify')
print(' stellar spectra — validated by SHAP XAI.')
print('=' * 60)

---
## Panel Defense Script

**Q: What is the research problem?**
> Neural networks classify stellar spectra accurately, but we don't know *which* spectral features they use. This is the **black-box problem** in astrophysics.

**Q: What does XAI do here?**
> SHAP (SHapley Additive exPlanations) assigns an **importance score to every wavelength point**. High SHAP values = wavelengths the model relied on most.

**Q: What did you discover?**
> The model's top features align with **known atomic absorption lines** (Fe I, Ti I, Mg I, Si I, O I) — proving the model learned real astrophysics, not noise. We quantify this with the **AAAS metric** (Absolute Average Attribution Score).

**Q: Why is this important?**
> Astronomers can now **trust** the model's predictions because the XAI explanation matches their domain knowledge. It also helps discover which lines are most diagnostically useful.
